# PICKO Research · NB1 — **Breadth**: how many tools before it breaks?

Finetune a **separate model per tool-set size** (nested), offer tools in **compact** form
(name + description, no parameters — so more names fit the encoder), and measure **tool selection only**.
The 1024-token encoder truncates the offered list, so past ~20 tools some are never seen — that ceiling
is the result, annotated with `n_visible`.

## 0 · Colab quick-start (GPU) — run & forget, restart-safe

**On Colab: Runtime → Change runtime type → GPU (T4) first.** This cell clones the repo, pins the exact
JAX/Flax, mounts Drive (so checkpoints survive a restart), and sets the output dir. **Running locally?**
It's a no-op — just skip to cell 1.

In [ ]:
# --- Colab bootstrap (safe to re-run; no-op locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/picko"):
        !git clone -b hadar-work https://github.com/HadarBit/picko.git /content/picko
    %pip install -q "jax[cuda12]==0.10.2" "jaxlib==0.10.2" "flax==0.12.8"
    sys.path.insert(0, "/content/picko")
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["PICKO_OUT_DIR"] = "/content/drive/MyDrive/picko_out"; os.makedirs(os.environ["PICKO_OUT_DIR"], exist_ok=True)
    import shutil
    src, dst = "/content/drive/MyDrive/picko_balanced.jsonl", "/content/picko/data/picko_balanced.jsonl"
    if os.path.exists(src) and not os.path.exists(dst): shutil.copy(src, dst)
    print("GPU:")
    !nvidia-smi -L
    assert os.path.exists(dst), "Data missing: it ships in the repo clone; if absent, upload picko_balanced.jsonl to /content/picko/data/ or Drive root."
    print("bootstrap OK · OUT_DIR =", os.environ["PICKO_OUT_DIR"])
else:
    print("Not on Colab — running locally (CPU).")

## 1 · Setup & data overview

In [ ]:
# ensure the repo root is importable (works from notebooks/research/, Colab, etc.)
import os, sys
_here = os.path.abspath(os.getcwd())
for _ in range(6):
    if os.path.exists(os.path.join(_here, "scripts", "picko_research.py")): break
    _here = os.path.dirname(_here)
if os.path.isdir("/content/picko"): _here = "/content/picko"
if _here not in sys.path: sys.path.insert(0, _here)

from scripts.picko_research import *
import pandas as pd, numpy as np, matplotlib.pyplot as plt
try:
    import seaborn as sns; sns.set_theme(style="whitegrid")
except Exception:
    sns = None
from tqdm.auto import tqdm

cat, tok, raw, FOCUS, OUT_DIR = load_context()

### The 40 focus tools\nOne row per tool, with its family, category and **parameter count / bucket**.

In [ ]:
display(tools_dataframe(cat, FOCUS))

### All examples for these 40 tools\nOne row per training example (query → gold tool), tagged with the gold tool's **param bucket**.

In [ ]:
ex_df = examples_dataframe(cat, raw, FOCUS)
print("examples:", ex_df.shape[0], "| per param bucket:", ex_df["param_bucket"].value_counts().to_dict())
display(ex_df.head(10))

## 2 · Configure the sweep\nEdit `BREADTH_SIZES` to change the tool counts tested. Sizes ≤ 40 stay inside the focus; larger sizes pull extra tools from the full 75-catalog.

In [ ]:
BREADTH_SIZES  = [3, 5, 10, 20, 30, 40]   # <- edit me
CAP_PER_TOOL   = 40      # examples/tool per finetune (raise to 120 for higher fidelity)
EPOCHS         = 1
EVAL_SUBSAMPLE = 30      # cap test examples per run for faster eval; None = full
RUN_TRAIN      = True
FORCE_RETRAIN  = False   # True = retrain even if a checkpoint exists

pool = breadth_pool(cat, FOCUS, seed=0)
SETS = size_sets(pool, BREADTH_SIZES)
print({k: len(v) for k, v in SETS.items()})

## 3 · Finetune per size & evaluate selection

In [ ]:
rows = []
for k in BREADTH_SIZES:
    names = SETS[k]
    R = finetune_and_eval(cat, raw, tok, names, f"breadth_k{k}", OUT_DIR,
                          cap=CAP_PER_TOOL, epochs=EPOCHS, compact=True, offer_all=k,
                          eval_subsample=EVAL_SUBSAMPLE, run_train=RUN_TRAIN, force_retrain=FORCE_RETRAIN)
    vis = int(np.median([n_visible(e["query"], json.loads(e["tools"]), tok) for e in R["test"]]))
    rows.append({"k": k, "selection_acc": R["metrics"]["selection_acc"],
                 "name_f1": R["metrics"]["name_f1"], "n_visible": vis, "n_test": len(R["test"])})
    print(f"  k={k}: selection={R['metrics']['selection_acc']:.3f} visible={vis}/{k}")
breadth = pd.DataFrame(rows)
# persist for later viewing
import json as _json
_json.dump(rows, open(os.path.join(OUT_DIR, "breadth_results.json"), "w"), indent=2)
display(breadth)

## 4 · The Breadth curve

In [ ]:
fig, ax = plt.subplots(figsize=(8,4.5))
ax.plot(breadth["k"], breadth["selection_acc"], "o-", color="#4C72B0", label="selection_acc")
ax.plot(breadth["k"], breadth["name_f1"], "s--", color="#55A868", label="name_f1")
wall = breadth[breadth["n_visible"] < breadth["k"]]
if len(wall):
    kw = int(wall["k"].iloc[0]); vw = int(wall["n_visible"].iloc[0])
    ax.axvline(kw, color="#C44E52", ls=":", lw=1.5)
    ax.text(kw, 0.06, f" truncation wall\n (~{vw} of {kw} tools visible)", color="#C44E52", fontsize=9, va="bottom")
ax.set_xlabel("# tools trained / offered (k)"); ax.set_ylabel("tool-selection accuracy")
ax.set_ylim(0,1.02); ax.set_title("Breadth: selection accuracy vs tool-set size"); ax.legend()
plt.tight_layout(); plt.show()

## 5 · Read-out

- Selection holds up to ~`k` tools then drops; the red line marks where the **compact** offered list stops
  fitting the 1024-token encoder (so the extra tools are truncated away and can't be picked).
- **Takeaway:** one PICKO instance is bounded by the *context window*, not raw capacity — beyond the wall,
  a large tool set should be sharded across categorical instances.